# M3 – Step 6: Request 2 results, noise, and refit on 24 points

1. Did request 2 behave as the models predicted? (a true hold-out check: these points were not used for fitting)
2. Noise from the repeat
3. Refit all surrogates on all data, LOOCV again (verification table + parity plot)
4. GP mean and σ maps, FM vs T slices with the new points
5. What this means for request 3 (6 points left)

Repeated (D, T) are averaged into one point before fitting (`average_repeats`), so interpolating models work and
LOO is not optimistic. For every section: **prediction before running**, **observation** after.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

ROOT = Path.cwd()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from m3_surrogate import (load_oracle, average_repeats, make_features, disk_loading, QuadraticSurrogate,
                          RBFSurrogate, GPSurrogate, loo_predictions, loo_table, D_MIN, D_MAX, T_MIN, T_MAX)

ORACLE_DIR = ROOT / 'data' / 'oracle'
FIG_DIR = ROOT / 'figures'; RES_DIR = ROOT / 'results'
pd.set_option('display.width', 200)
raw = load_oracle(ORACLE_DIR)
print(raw.groupby('batch').size())

## 1. Request 2: prediction vs result (hold-out check)

`results/m3_request2_plan.csv` holds what GP and RBF cubic predicted **before** the oracle answered.
z = (FM − GP mean)/GP σ: |z| > 2 means the GP was over-confident at that point.

Prediction: _..._

In [ ]:
plan = pd.read_csv(RES_DIR / 'm3_request2_plan.csv', index_col=0)
r2 = raw[raw.batch == 'result_2'].reset_index(drop=True)
assert np.allclose(plan[['D_m', 'T_N']].values, r2[['D_m', 'T_N']].values), "result_2 rows do not match request 2"
plan['FM'] = r2.FM.values
plan['err_GP'] = plan.FM - plan.FM_GP
plan['z_GP'] = plan.err_GP / plan.sd_GP.clip(lower=1e-3)
plan['err_RBF'] = plan.FM - plan.FM_RBF
show = ['category', 'D_m', 'T_N', 'DL', 'FM', 'FM_GP', 'sd_GP', 'err_GP', 'z_GP', 'FM_RBF', 'err_RBF']
display(plan[show].round(3))

new = plan.category != 'repeat'
for name, col in [('GP (D,T)', 'err_GP'), ('RBF cubic', 'err_RBF')]:
    e = plan.loc[new, col]
    print(f"{name:10s} hold-out RMSE on 8 new points = {np.sqrt(np.mean(e**2)):.4f}"
          f"   without the DL-416 corner = {np.sqrt(np.mean(e[plan.DL[new] < 300]**2)):.4f}")

Observation (which model predicted the new points better? what happened at (0.35 m, 40 N)?): _..._

## 2. Noise from the repeat

One repeated point gives one difference d. If the oracle noise is Gaussian with std σ_n, the difference of two
runs has std √2·σ_n, so a rough estimate is σ_n ≈ |d|/√2. One pair is weak evidence: treat it as an order of magnitude.

Prediction: _..._

In [ ]:
df, repeats = average_repeats(raw)
print(repeats.to_string(index=False))
d = repeats['diff'].values
sigma_n = np.abs(d) / np.sqrt(2)
print(f"\nnoise std estimate sigma_n ≈ {sigma_n.mean():.4f}   (variance {sigma_n.mean()**2:.2e})")
print(f"std of FM over all data = {df.FM.std(ddof=1):.4f}")

Observation (is the noise small compared with the LOO errors below? so is the misfit noise or model shape?): _..._

## 3. Refit on all unique points and LOOCV

Same five models as step 3/4, inputs scaled (D, T). One extra **reference** row: the same GP on
(scaled log DL, scaled D), the inputs suggested earlier and rejected for simplicity, to check that trade-off with
more data (logged as "revisit after request 2").

Prediction: _..._

In [ ]:
X, y = make_features(df.D_m, df.T_N), df.FM.values
models = {
    'quadratic': QuadraticSurrogate(),
    'RBF thin-plate exact': RBFSurrogate('thin_plate_spline', 0.0),
    'RBF thin-plate s=1e-3': RBFSurrogate('thin_plate_spline', 1e-3),
    'RBF cubic': RBFSurrogate('cubic', 0.0),
    'GP (ARD + nugget)': GPSurrogate(),
}
loo = {k: loo_predictions(m, X, y) for k, m in models.items()}

# reference: GP on (log DL, D), scaled over the oracle box corners
lo, hi = np.log(disk_loading(D_MAX, T_MIN)), np.log(disk_loading(D_MIN, T_MAX))
def feat_logdl(D, T):
    D, T = np.atleast_1d(D), np.atleast_1d(T)
    return np.column_stack([(np.log(disk_loading(D, T)) - lo) / (hi - lo), (D - D_MIN) / (D_MAX - D_MIN)])
XL = feat_logdl(df.D_m.values, df.T_N.values)
loo['GP on (log DL, D) [reference]'] = loo_predictions(GPSurrogate(), XL, y)

table = loo_table(loo, y)
table.to_csv(RES_DIR / 'm3_loo_table_24pts.csv', float_format='%.4f')
display(table.round(4))
print(f"std of FM = {y.std(ddof=1):.4f};  pt numbers refer to the unique-point table below")
display(df[['D_m', 'T_N', 'DL', 'FM', 'n', 'batch']].round(3))

In [ ]:
gp = GPSurrogate().fit(X, y)
gpl = GPSurrogate().fit(XL, y)
hp = pd.DataFrame({'GP (D, T)': gp.hyperparameters(), 'GP (log DL, D)': gpl.hyperparameters()})
hp.index = [i.replace('length_D', 'length input 1').replace('length_T', 'length input 2') for i in hp.index]
print("input 1 / input 2 = D / T for the (D, T) GP, log DL / D for the reference GP")
display(hp.round(5))

Observation (ranking now vs request 1; is the GP still over-confident (mean z²)? in the reference GP, how long is
the D length scale, i.e. does D matter once DL is known?): _..._

In [ ]:
names = list(loo)
fig, axes = plt.subplots(2, 3, figsize=(13, 8.5), sharex=True, sharey=True)
lo_, hi_ = y.min() - 0.05, y.max() + 0.05
for ax, name in zip(axes.flat, names):
    mu, sd = loo[name]
    if np.all(np.isfinite(sd)):
        ax.errorbar(y, mu, yerr=2 * sd, fmt='none', ecolor='grey', alpha=0.6)
    sc = ax.scatter(y, mu, c=df.DL, norm=LogNorm(), cmap='plasma', edgecolor='k', s=50, zorder=3,
                    marker='o')
    r2m = df.batch.str.contains('result_2').values
    ax.scatter(y[r2m], mu[r2m], facecolor='none', edgecolor='lime', s=120, lw=1.5, zorder=4)
    for i in df.index:
        ax.annotate(str(i), (y[i-1], mu[i-1]), xytext=(4, 3), textcoords='offset points', fontsize=7)
    ax.plot([lo_, hi_], [lo_, hi_], 'k--', lw=1)
    r = table.loc[name]
    ax.set_title(f"{name}\nLOO RMSE={r['LOO RMSE']:.4f}  Q²={r['LOO Q2']:.2f}", fontsize=9)
    ax.grid(alpha=0.3); ax.set_xlim(lo_, hi_); ax.set_ylim(lo_, hi_)
for ax in axes[-1]: ax.set_xlabel('Oracle FM')
for ax in axes[:, 0]: ax.set_ylabel('LOO prediction')
fig.colorbar(sc, ax=axes, label='DL (N/m²)', shrink=0.8)
fig.suptitle('M3 verification after request 2 (24 runs, 23 unique points): LOO parity; green ring = request-2 point')
fig.savefig(FIG_DIR / 'm3_loo_parity_24pts.png', dpi=150, bbox_inches='tight'); plt.show()

Observation: _..._

## 4. Surrogates over the box

**4a.** GP (D, T) mean and σ, with all points and M1 designs. **4b.** FM vs T at three diameters for RBF cubic,
GP (D, T) and the reference GP (log DL, D).

Prediction (where is σ still largest? does any model follow the cliff at small D, high T?): _..._

In [ ]:
designs = pd.read_csv(RES_DIR / 'm1_designs.csv')
Dg, Tg = np.meshgrid(np.linspace(D_MIN, D_MAX, 150), np.linspace(T_MIN, T_MAX, 150))
mu_g, sd_g = gp.predict(make_features(Dg.ravel(), Tg.ravel()), return_std=True)
mu_g, sd_g = mu_g.reshape(Dg.shape), sd_g.reshape(Dg.shape)
DLg = disk_loading(Dg, Tg)
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, Z, title, cmap in [(axes[0], mu_g, 'GP (D, T) mean FM, 23 points', 'viridis'),
                           (axes[1], sd_g, 'GP (D, T) σ, 23 points', 'magma')]:
    cf = ax.contourf(Dg, Tg, Z, levels=20, cmap=cmap); plt.colorbar(cf, ax=ax)
    cs = ax.contour(Dg, Tg, DLg, levels=[25, 50, 100, 151, 217, 300], colors='w', linewidths=0.7, linestyles='--')
    ax.clabel(cs, fmt='DL=%d', fontsize=7)
    for b, mk in [('result_1', 'o'), ('result_2', 's')]:
        s = df[df.batch.str.contains(b)]
        ax.scatter(s.D_m, s.T_N, marker=mk, c='w', edgecolor='k', s=50, zorder=3, label=b)
    for _, r in designs.iterrows():
        ax.plot([r.D, r.D], [r.T_hover_empty, r.T_hover_loaded], '-', color='cyan', lw=2)
        ax.annotate(r.design.split()[0], (r.D, r.T_hover_loaded), xytext=(-10, 3), textcoords='offset points',
                    color='cyan', fontweight='bold')
    ax.set_xlabel('D (m)'); ax.set_ylabel('T (N)'); ax.set_title(title)
axes[0].legend(loc='lower right', fontsize=8)
plt.tight_layout(); fig.savefig(FIG_DIR / 'm3_gp_mean_std_24pts.png', dpi=150); plt.show()
k = np.unravel_index(np.argmax(sd_g), sd_g.shape)
print(f"max σ = {sd_g[k]:.4f} at D = {Dg[k]:.3f} m, T = {Tg[k]:.1f} N (DL {DLg[k]:.0f})")

In [ ]:
rbf = RBFSurrogate('cubic').fit(X, y)
T_line = np.linspace(T_MIN, T_MAX, 300)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), sharey=True)
for ax, Ds in zip(axes, [0.38, 0.575, 0.75]):
    D_line = np.full_like(T_line, Ds)
    Xs = make_features(D_line, T_line)
    m, s = gp.predict(Xs, return_std=True)
    ax.fill_between(T_line, m - 2*s, m + 2*s, color='grey', alpha=0.3, label='GP (D,T) ±2σ')
    ax.plot(T_line, m, 'r', label='GP (D, T)')
    ax.plot(T_line, rbf.predict(Xs), 'g', label='RBF cubic')
    ax.plot(T_line, gpl.predict(feat_logdl(D_line, T_line)), 'b--', label='GP (log DL, D) ref.')
    near = np.abs(df.D_m - Ds) < 0.05
    ax.scatter(df.T_N[near], df.FM[near], facecolor='none', edgecolor='k', s=90, zorder=5, label='data |ΔD|<0.05')
    ax.set_title(f'D = {Ds} m'); ax.set_xlabel('T (N)'); ax.grid(alpha=0.3)
axes[0].set_ylabel('FM'); axes[0].set_ylim(0.1, 0.8); axes[0].legend(fontsize=8)
plt.tight_layout(); fig.savefig(FIG_DIR / 'm3_slices_24pts.png', dpi=150); plt.show()

Observation (between T ≈ 26 and 40 N at small D, the models have no data: how different are they?): _..._

## 5. Request 3 (6 points left): your plan

Use the table and maps above. Points to think about:
- the cliff: nothing measured between DL 225 (FM 0.47) and DL 416 (FM 0.20)
- the M1 re-run with the surrogate FM will move designs A–E: their new hover points are exploitation targets
- noise looks small (one repeat), so another repeat may be worth less than a new location
- random points were a useful unbiased check on the models

My plan: _..._